# Lesson 08 — Exercise Solutions

Worked solutions with a short rationale after each. Run setup.sql first.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

Insert 3 new rows into lesson08.ProductStaging with ProductIDs 8001, 8002, 8003, names 'Test A', 'Test B', 'Test C', and ListPrices 10.00, 20.00, 30.00.

In [ ]:
INSERT lesson08.ProductStaging (ProductID, Name, ListPrice)
VALUES (8001, N'Test A', 10.00),
       (8002, N'Test B', 20.00),
       (8003, N'Test C', 30.00);

**Why:** multi-row VALUES list with explicit columns.

## Exercise 2

Update the ListPrice of ALL rows in lesson08.ProductStaging where ProductID >= 8001 to ListPrice * 2 (double them). Use OUTPUT to return the ProductID, old price, and new price.
Expected columns in output: ProductID, OldPrice, NewPrice

In [ ]:
UPDATE lesson08.ProductStaging
SET    ListPrice = ListPrice * 2
OUTPUT inserted.ProductID,
       deleted.ListPrice  AS OldPrice,
       inserted.ListPrice AS NewPrice
WHERE  ProductID >= 8001;

**Why:** OUTPUT deleted.ListPrice (before) and inserted.ListPrice (after).

## Exercise 3

Delete all rows from lesson08.ProductStaging where ListPrice = 0. Use OUTPUT to capture the deleted ProductIDs and Names.

In [ ]:
DELETE lesson08.ProductStaging
OUTPUT deleted.ProductID, deleted.Name
WHERE  ListPrice = 0;

**Why:** OUTPUT deleted.* captures the rows being removed.

## Exercise 4

Write a MERGE statement that uses lesson08.InventorySource as the source and lesson08.InventoryTarget as the target.
- WHEN MATCHED AND stock differs: update StockQty and LastSyncAt
- WHEN NOT MATCHED BY TARGET: insert the new row
- WHEN NOT MATCHED BY SOURCE: do nothing (no DELETE this time)

In [ ]:
MERGE lesson08.InventoryTarget AS t
USING lesson08.InventorySource AS s
   ON s.ProductID = t.ProductID
WHEN MATCHED AND s.StockQty <> t.StockQty THEN
    UPDATE SET t.StockQty = s.StockQty, t.LastSyncAt = SYSDATETIME()
WHEN NOT MATCHED BY TARGET THEN
    INSERT (ProductID, StockQty, LastSyncAt)
    VALUES (s.ProductID, s.StockQty, SYSDATETIME());

**Why:** omit WHEN NOT MATCHED BY SOURCE to leave orphan target rows alone.

## Exercise 5

Using INSERT...SELECT and OUTPUT, copy all lesson08.ProductStaging rows with ListPrice > 50 into a table variable, then SELECT from it.
Expected columns (in table variable and final SELECT): ProductID, Name, ListPrice

In [ ]:
DECLARE @Copied TABLE (ProductID INT, Name NVARCHAR(50), ListPrice MONEY);

INSERT lesson08.ProductStaging (ProductID, Name, ListPrice)
OUTPUT inserted.ProductID, inserted.Name, inserted.ListPrice
INTO   @Copied
SELECT ProductID, Name, ListPrice
FROM   lesson08.ProductStaging
WHERE  ListPrice > 50
  AND  ProductID NOT IN (SELECT ProductID FROM lesson08.ProductStaging WHERE ProductID > 9000);
-- Note: this would insert duplicates in practice; the exercise focuses on the OUTPUT syntax.

SELECT * FROM @Copied;

**Why:** declare the table variable first; use OUTPUT INTO to fill it.

## Challenges

Harder problems. They may combine this lesson's topic with
earlier lessons, and usually need 2+ concepts in one query.

### Challenge 1

Idempotent load: insert into the lesson08 target only the staging rows not already present (key match), capturing inserted keys via OUTPUT into a table variable and selecting them. Run it twice — second run must insert 0 rows.
Expected columns: (inserted keys result set)
Hint: INSERT ... SELECT ... WHERE NOT EXISTS.

In [ ]:
DECLARE @Inserted1 TABLE (ProductID INT);
INSERT lesson08.ProductPrice (ProductID, ListPrice, ModifiedDate)
OUTPUT inserted.ProductID INTO @Inserted1
SELECT s.ProductID, s.ListPrice, s.ModifiedDate
FROM lesson08.ProductPriceStage AS s
WHERE NOT EXISTS (
    SELECT 1 FROM lesson08.ProductPrice AS t WHERE t.ProductID = s.ProductID
);
SELECT * FROM @Inserted1;  -- run 1: ProductID 4 and 5

DECLARE @Inserted2 TABLE (ProductID INT);
INSERT lesson08.ProductPrice (ProductID, ListPrice, ModifiedDate)
OUTPUT inserted.ProductID INTO @Inserted2
SELECT s.ProductID, s.ListPrice, s.ModifiedDate
FROM lesson08.ProductPriceStage AS s
WHERE NOT EXISTS (
    SELECT 1 FROM lesson08.ProductPrice AS t WHERE t.ProductID = s.ProductID
);
SELECT * FROM @Inserted2;                              -- run 2: no rows
SELECT COUNT(*) AS SecondRunInsertCount FROM @Inserted2; -- must be 0

**Why:** the tempting-but-wrong approach is a plain INSERT...SELECT of every staging row, which duplicates on every re-run since ProductID isn't checked. NOT EXISTS against the target on the natural key filters to genuinely new rows only, so re-running finds nothing left to insert. OUTPUT INTO a table variable captures which keys landed without a second SELECT. Target starts with 1, 2, 3; staging has 1, 2, 4, 5 — so run 1 inserts 4 and 5, and run 2 (below) inserts nothing.

### Challenge 2

Full sync with audit: MERGE staging into target (update changed, insert new, delete missing) writing $action plus old/new values via OUTPUT into an audit table. Verify the audit row counts match the changes you staged.
Expected columns: (audit table contents)
Hint: OUTPUT $action, deleted.*, inserted.*.

In [ ]:
INSERT lesson08.ProductPriceStage (ProductID, ListPrice, ModifiedDate)
SELECT 7, 70.00, SYSDATETIME()
WHERE NOT EXISTS (SELECT 1 FROM lesson08.ProductPriceStage WHERE ProductID = 7);

MERGE lesson08.ProductPrice AS t
USING lesson08.ProductPriceStage AS s
   ON s.ProductID = t.ProductID
WHEN MATCHED AND s.ListPrice <> t.ListPrice THEN
    UPDATE SET t.ListPrice = s.ListPrice, t.ModifiedDate = s.ModifiedDate
WHEN NOT MATCHED BY TARGET THEN
    INSERT (ProductID, ListPrice, ModifiedDate)
    VALUES (s.ProductID, s.ListPrice, s.ModifiedDate)
WHEN NOT MATCHED BY SOURCE THEN
    DELETE
OUTPUT $action, COALESCE(inserted.ProductID, deleted.ProductID), deleted.ListPrice, inserted.ListPrice
INTO lesson08.ProductPriceAudit (Action, ProductID, OldPrice, NewPrice);

-- Verify: one audit row per staged change (update/insert/delete all represented)
SELECT Action, ProductID, OldPrice, NewPrice FROM lesson08.ProductPriceAudit ORDER BY AuditID;
SELECT Action, COUNT(*) AS Cnt FROM lesson08.ProductPriceAudit GROUP BY Action;

**Why:** MERGE is the right tool since it needs update+insert+delete in one atomic statement sharing a single OUTPUT stream; three separate statements can't do that. An extra staging row (ProductID 7) guarantees a genuine INSERT case; it's guarded by NOT EXISTS so re-running the whole file doesn't hit a PK violation (same idempotent habit Challenge 1 teaches). $action plus COALESCE(inserted.*, deleted.*) — only one side is populated per action — go straight into the audit table via OUTPUT INTO.

### Challenge 3

Dedupe in place: seed duplicate rows (same natural key, different ModifiedDate) in a lesson08 table, then delete all but the newest per key using a CTE over ROW_NUMBER — a windowed DELETE. Prove with a count per key = 1.
Expected columns: (post-delete verification counts)
Hint: DELETE FROM cte WHERE rn > 1.

In [ ]:
INSERT lesson08.ProductPriceLog (ProductID, ListPrice, ModifiedDate)
VALUES (101, 10.00, '2026-01-01'),
       (101, 10.50, '2026-02-01'),
       (101, 11.00, '2026-03-01'),
       (102, 20.00, '2026-01-15'),
       (102, 21.00, '2026-02-15');

;WITH Ranked AS (
    SELECT LogID, ProductID,
           ROW_NUMBER() OVER (PARTITION BY ProductID ORDER BY ModifiedDate DESC) AS rn
    FROM lesson08.ProductPriceLog
)
DELETE FROM Ranked WHERE rn > 1;

-- Verify: exactly one surviving row per key
SELECT ProductID, COUNT(*) AS RowsPerKey
FROM lesson08.ProductPriceLog
GROUP BY ProductID
ORDER BY ProductID;

**Why:** the tempting-but-wrong approach is GROUP BY + MIN(LogID) to find "the" survivor, which can't express "keep the newest by ModifiedDate" without a second correlated query. ROW_NUMBER() PARTITION BY the natural key, ORDER BY ModifiedDate DESC numbers the newest row 1 per key; deleting through the CTE where rn > 1 is a windowed DELETE — you can't put a window function directly in DELETE's WHERE (lesson 07 pitfall), so the CTE is required.

### Challenge 4

Batch delete: delete a large set in chunks of 500 rows in a WHILE loop (using TOP), reporting rows deleted per iteration, and explain in the comment why chunking beats one giant DELETE (lock escalation, log growth — lessons 16/17 preview).
Expected columns: (iteration report via PRINT/SELECT)
Hint: WHILE @@ROWCOUNT ... DELETE TOP (500).

In [ ]:
INSERT lesson08.PurgeCandidate (Note)
SELECT TOP (2200) 'purge-' + CAST(ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS VARCHAR(10))
FROM sys.all_objects AS a CROSS JOIN sys.all_objects AS b;

DECLARE @RowsDeleted INT = 1, @BatchNum INT = 0;
WHILE @RowsDeleted > 0
BEGIN
    SET @BatchNum += 1;
    DELETE TOP (500) FROM lesson08.PurgeCandidate;
    SET @RowsDeleted = @@ROWCOUNT;
    PRINT CONCAT('Batch ', @BatchNum, ': deleted ', @RowsDeleted, ' rows');
END

SELECT COUNT(*) AS RemainingRows FROM lesson08.PurgeCandidate;  -- must be 0

**Why:** one giant DELETE holds its locks (risking escalation to table level) and keeps its whole log record un-reusable for the entire statement, blocking concurrent readers/writers and risking log growth. Deleting TOP (500) at a time in a loop lets each batch's locks and log space free up before the next batch starts — the same trade-off lessons 16 (locking/blocking) and 17 (waits) cover in depth; this is the practitioner's preview.